In [1]:
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from pathlib import Path
import wandb

REPO = Path("C:/Users/ameli/Desktop/TezBaselines")
sys.path.insert(0, str(REPO))

from DecompDiff.models.decompDiff import DecompDiff
from DecompDiff.models.diffusion  import GaussianDiffusion
from DecompDiff.config.stocks_config import Config
from MyCode.utils.data_utils.loader import create_data_loaders
from MyCode.eval_metrics import evaluate_samples, vds_score, fdds_score, correlational_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


In [2]:
def compute_loss(model, diffusion, x_0, loss_type="l1"):
    B = x_0.shape[0]
    t = torch.randint(0, diffusion.num_timesteps, (B,), device=x_0.device)
    x_t, _ = diffusion.q_sample(x_0, t)
    x_0_pred = model(x_t, t)
    loss = F.l1_loss(x_0_pred, x_0, reduction="none") if loss_type == "l1" \
           else F.mse_loss(x_0_pred, x_0, reduction="none")
    return (loss.mean(dim=[1, 2]) * diffusion.loss_weight[t]).mean()


def compute_inline_metrics(model, diffusion, real_loader, device, num_steps=50):
    # DiffusionTS protocol: ALL real windows, fake count = N, no cap.
    # Real = the same windows the model trained on (train_split=1.0).
    # The metric's internal 80/20 split then yields ~731 test points/class.
    model.eval()

    # ── collect ALL real windows ──────────────────────────────────────────────
    batches = []
    for b in real_loader:
        batches.append(b.cpu().numpy())
    real_CL = np.concatenate(batches, axis=0)   # (N, C, L) — every window
    N = real_CL.shape[0]

    # ── generate N fake windows (one per real window) ─────────────────────────
    chunk = 256
    chunks = []
    with torch.no_grad():
        for start in range(0, N, chunk):
            bs = min(chunk, N - start)
            chunks.append(
                model.sample(diffusion, batch_size=bs, num_steps=num_steps, eta=0.0).cpu()
            )
    fake_CL = torch.cat(chunks, dim=0).numpy()  # (N, C, L)

    # ── (N, C, L) [-1,1]  →  (N, L, C) [0,1] ───────────────────────────────
    real_m = ((real_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)
    fake_m = ((fake_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)

    # ── metrics (same as DiffusionTS protocol, PyTorch implementation) ─────────
    try:
        results = evaluate_samples(real_m, fake_m, device=device,
                                   n_iterations=3,
                                   disc_iterations=2000,
                                   pred_iterations=5000)
        vds  = vds_score(real_m, fake_m)
        fdds = fdds_score(real_m, fake_m)
        corr = correlational_score(real_m, fake_m)
    except Exception as e:
        print(f"  [metrics] failed: {e}")
        model.train()
        return None

    model.train()
    return {
        "disc_score":          results["discriminative"]["mean"],
        "disc_score_std":      results["discriminative"]["std"],
        "test_acc":            results["discriminative"]["test_acc"],
        "pred_mae":            results["predictive"]["mean"],
        "pred_mae_std":        results["predictive"]["std"],
        "vds":                 vds,
        "fdds":                fdds,
        "correlational_score": corr,
    }


def run_experiment(window_length, num_epochs, num_layers=None, hidden_dim=None, eval_every=100, run_name=None):
    cfg = Config()
    cfg.model.sequence_length = window_length
    cfg.training.num_epochs   = num_epochs
    if num_layers is not None:
        cfg.model.num_layers = num_layers
    if hidden_dim is not None:
        cfg.model.hidden_dim = hidden_dim
    # train_split stays at config default (1.0) — all data used for training

    run_name = run_name or (
        f"decompdiff-L{window_length}-H{cfg.model.hidden_dim}"
        f"-NL{cfg.model.num_layers}-E{num_epochs}"
    )

    wandb.init(
        project = "decompdiff",
        name    = run_name,
        config  = {
            **cfg.model.__dict__,
            **cfg.diffusion.__dict__,
            **cfg.training.__dict__,
            "device":     DEVICE,
            "eval_every": eval_every,
        },
    )

    # ── data (train_split=1.0 from config — no val set) ───────────────────────
    train_loader, _, _ = create_data_loaders(
        csv_path       = cfg.data.data_path,
        batch_size     = cfg.training.batch_size,
        window_length  = window_length,
        neg_one_to_one = cfg.data.neg_one_to_one,
        train_ratio    = cfg.data.train_split,
        num_workers    = cfg.data.num_workers,
        per_window     = cfg.data.per_window_norm,
        pin_memory     = False,
    )
    print(f"[{run_name}] train batches: {len(train_loader)}  (train_split={cfg.data.train_split})")

    # ── model ─────────────────────────────────────────────────────────────────
    model = DecompDiff(
        input_channels  = cfg.model.input_channels,
        sequence_length = window_length,
        hidden_dim      = cfg.model.hidden_dim,
        num_heads       = cfg.model.num_heads,
        num_layers      = cfg.model.num_layers,
        mlp_ratio       = cfg.model.mlp_ratio,
        dropout         = cfg.model.dropout,
        freq_dim        = cfg.model.freq_dim,
    ).to(DEVICE)

    diffusion = GaussianDiffusion(
        num_timesteps  = cfg.diffusion.num_timesteps,
        beta_start     = cfg.diffusion.beta_start,
        beta_end       = cfg.diffusion.beta_end,
        noise_schedule = cfg.diffusion.noise_schedule,
        device         = DEVICE,
    ).to(DEVICE)

    counts = model.get_parameter_count()
    print(f"[{run_name}] params: {counts['total']:,}")
    wandb.config.update({"total_params": counts["total"]}, allow_val_change=True)

    # ── optimiser ─────────────────────────────────────────────────────────────
    total_steps = len(train_loader) * num_epochs
    optimizer = AdamW(model.parameters(), lr=cfg.training.learning_rate,
                      weight_decay=cfg.training.weight_decay, betas=(0.9, 0.999))
    warmup = LinearLR(optimizer, start_factor=1e-3, end_factor=1.0,
                      total_iters=cfg.training.warmup_steps)
    cosine = CosineAnnealingLR(optimizer,
                               T_max=max(1, total_steps - cfg.training.warmup_steps),
                               eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                              milestones=[cfg.training.warmup_steps])

    ckpt_dir = REPO / f"DecompDiff/output/checkpoints/{run_name}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    save_every = max(100, num_epochs // 5)   # save 5 checkpoints during training

    # ── training loop ─────────────────────────────────────────────────────────
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            x_0 = batch.to(DEVICE)
            optimizer.zero_grad()
            loss = compute_loss(model, diffusion, x_0, cfg.training.loss_type)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.training.gradient_clip_val)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        train_loss = epoch_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]["lr"]
        log = {"train_loss": train_loss, "lr": current_lr, "epoch": epoch + 1}

        # checkpoint every save_every epochs and at the final epoch
        if (epoch + 1) % save_every == 0 or (epoch + 1) == num_epochs:
            torch.save({
                "epoch":            epoch + 1,
                "train_loss":       train_loss,
                "model_state_dict": model.state_dict(),
                "config":           {"model": cfg.model.__dict__,
                                     "window_length": window_length},
            }, ckpt_dir / f"checkpoint_ep{epoch+1}.pt")

        # inline metrics every eval_every epochs — all windows (DiffusionTS protocol)
        if (epoch + 1) % eval_every == 0:
            print(f"  [epoch {epoch+1}] computing metrics (all windows)...")
            metrics = compute_inline_metrics(
                model, diffusion, train_loader, DEVICE, num_steps=50,
            )
            if metrics is not None:
                log.update(metrics)
                print(
                    f"  disc={metrics['disc_score']:.4f} "
                    f"pred={metrics['pred_mae']:.4f} "
                    f"vds={metrics['vds']:.4f} "
                    f"fdds={metrics['fdds']:.4f} "
                    f"corr={metrics['correlational_score']:.4f}"
                )

        wandb.log(log)
        print(f"[{run_name}] ep {epoch+1:4d}/{num_epochs}  "
              f"train={train_loss:.5f}  lr={current_lr:.2e}")

    print(f"[{run_name}] done.  ckpt -> {ckpt_dir}")
    wandb.finish()
    return model, diffusion

In [ ]:
# ── Experiment 1: window = 24, num_layers = 1 (default) ──────────────────────
model_24, diff_24 = run_experiment(window_length=24, num_epochs=2000, eval_every=400)

StockDataset: 3662 windows  (train=3662, test=all [train_ratio=1.0])
[decompdiff-L24-NL1-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L24-NL1-E2000] params: 652,550
[decompdiff-L24-NL1-E2000] ep    1/2000  train=1.03853  lr=5.80e-05
[decompdiff-L24-NL1-E2000] ep    2/2000  train=0.47844  lr=1.00e-04
[decompdiff-L24-NL1-E2000] ep    3/2000  train=0.23147  lr=1.00e-04
[decompdiff-L24-NL1-E2000] ep    4/2000  train=0.20826  lr=1.00e-04


In [ ]:
# ── Experiment 2: window = 32, num_layers = 1 (default) ──────────────────────
model_32, diff_32 = run_experiment(window_length=32, num_epochs=1000, eval_every=100)

In [ ]:
# ── Experiment 3: window = 32, num_layers = 2 ────────────────────────────────
model_32_nl2, diff_32_nl2 = run_experiment(window_length=32, num_epochs=1000, num_layers=2, eval_every=100)

In [ ]:
# ── Experiment 4: window = 32, num_layers = 4 ────────────────────────────────
model_32_nl4, diff_32_nl4 = run_experiment(window_length=32, num_epochs=1000, num_layers=4, eval_every=100)

In [3]:
# ── Experiment 5: window = 32, hidden_dim = 64 ───────────────────────────────
model_32_h64, diff_32_h64 = run_experiment(window_length=32, num_epochs=2000, hidden_dim=64, eval_every=500)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\ameli\_netrc.
wandb: Currently logged in as: a-meliksahdemir (a-meliksahdemir-bo-azi-i-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H64-NL1-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H64-NL1-E2000] params: 175,750
[decompdiff-L32-H64-NL1-E2000] ep    1/2000  train=1.21680  lr=5.80e-05
[decompdiff-L32-H64-NL1-E2000] ep    2/2000  train=0.80612  lr=1.00e-04
[decompdiff-L32-H64-NL1-E2000] ep    3/2000  train=0.28795  lr=1.00e-04
[decompdiff-L32-H64-NL1-E2000] ep    4/2000  train=0.22434  lr=1.00e-04
[decompdiff-L32-H64-NL1-E2000] ep    5/2000  train=0.20358  lr=1.00e-04
[decompdiff-L32-H64-NL1-E2000] ep    6/2000  train=0.19067  lr=1.00e-04
[decompdiff-L32-H64-NL1-E2000] ep    7/2000  train=0.18165  lr=1.00e-04
[decompdiff-L32-H64-NL1-E2000] ep    8/2000  train=0.17880  lr=1.00e-04
[decompdiff-L32-H64-NL1-E2000] ep    9/2000  train=0.17068  lr=1.00e-04
[decompdiff-L32-H64-NL1-E2000] ep   10/2000  train=0.16594  lr=1.00e-04
[decompdiff-L32-H64-NL1-E2000] ep   11/2000  train=0.16254  lr=1.00e-04
[decompdiff-L32-H64-NL1-

correlational_score,█▃▁▅
disc_score,█▃▅▁
disc_score_std,█▂▂▁
epoch,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇█████
fdds,▁▅▇█
lr,██████▇▇▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁
pred_mae,█▄▄▁
pred_mae_std,█▃▁▄
test_acc,█▃▅▁
train_loss,█▇█▆▇▅▅▃▄▅▄▄▄▃▂▃▄▃▃▃▃▃▃▃▃▃▃▃▂▃▂▁▂▂▃▂▂▁▂▁
+1,...


In [4]:
# ── Experiment 6: window = 32, hidden_dim = 256 ──────────────────────────────
model_32_h256, diff_32_h256 = run_experiment(window_length=32, num_epochs=2000, hidden_dim=256, eval_every=500)

StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H256-NL1-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H256-NL1-E2000] params: 2,521,606
[decompdiff-L32-H256-NL1-E2000] ep    1/2000  train=1.12617  lr=5.80e-05
[decompdiff-L32-H256-NL1-E2000] ep    2/2000  train=0.32869  lr=1.00e-04
[decompdiff-L32-H256-NL1-E2000] ep    3/2000  train=0.20610  lr=1.00e-04
[decompdiff-L32-H256-NL1-E2000] ep    4/2000  train=0.19162  lr=1.00e-04
[decompdiff-L32-H256-NL1-E2000] ep    5/2000  train=0.18213  lr=1.00e-04
[decompdiff-L32-H256-NL1-E2000] ep    6/2000  train=0.16729  lr=1.00e-04
[decompdiff-L32-H256-NL1-E2000] ep    7/2000  train=0.15760  lr=1.00e-04
[decompdiff-L32-H256-NL1-E2000] ep    8/2000  train=0.15371  lr=1.00e-04
[decompdiff-L32-H256-NL1-E2000] ep    9/2000  train=0.14767  lr=1.00e-04
[decompdiff-L32-H256-NL1-E2000] ep   10/2000  train=0.14547  lr=1.00e-04
[decompdiff-L32-H256-NL1-E2000] ep   11/2000  train=0.14552  lr=1.00e-04
[decompdi

correlational_score,▁▄█▆
disc_score,▆▁▇█
disc_score_std,▁▂██
epoch,▁▁▁▁▁▁▁▁▁▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇███
fdds,▁▂▆█
lr,█████████▇▇▇▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁
pred_mae,▁█▂▃
pred_mae_std,▁█▃▂
test_acc,▆▁▇█
train_loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▂▂▁▂▁▂▁▁▁▂▁▁▁▁▁▁▁▂▂▁
+1,...


In [5]:
# ── window = 32, hidden_dim = 64, num_layers = 2 ─────────────────────────────
model_32_h64_nl2, diff_32_h64_nl2 = run_experiment(
    window_length=32, num_epochs=2000, hidden_dim=64, num_layers=2, eval_every=500)


StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H64-NL2-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H64-NL2-E2000] params: 325,126
[decompdiff-L32-H64-NL2-E2000] ep    1/2000  train=1.05900  lr=5.80e-05
[decompdiff-L32-H64-NL2-E2000] ep    2/2000  train=0.65690  lr=1.00e-04
[decompdiff-L32-H64-NL2-E2000] ep    3/2000  train=0.26158  lr=1.00e-04
[decompdiff-L32-H64-NL2-E2000] ep    4/2000  train=0.21446  lr=1.00e-04
[decompdiff-L32-H64-NL2-E2000] ep    5/2000  train=0.19525  lr=1.00e-04
[decompdiff-L32-H64-NL2-E2000] ep    6/2000  train=0.17820  lr=1.00e-04
[decompdiff-L32-H64-NL2-E2000] ep    7/2000  train=0.16920  lr=1.00e-04
[decompdiff-L32-H64-NL2-E2000] ep    8/2000  train=0.16307  lr=1.00e-04
[decompdiff-L32-H64-NL2-E2000] ep    9/2000  train=0.15341  lr=1.00e-04
[decompdiff-L32-H64-NL2-E2000] ep   10/2000  train=0.14853  lr=1.00e-04
[decompdiff-L32-H64-NL2-E2000] ep   11/2000  train=0.14355  lr=1.00e-04
[decompdiff-L32-H64-NL2-

correlational_score,▂█▁▃
disc_score,▁██▃
disc_score_std,█▇▁▅
epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇█████
fdds,▁▃██
lr,█████▇▇▇▇▇▇▇▇▆▆▆▆▆▆▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
pred_mae,▃█▂▁
pred_mae_std,█▁▃▁
test_acc,▁██▃
train_loss,█▅▅▅▃▄▄▄▃▃▄▃▃▃▃▃▂▃▃▂▃▂▃▃▃▃▃▂▁▂▂▃▂▂▂▂▂▂▁▂
+1,...


In [6]:
# ── window = 32, hidden_dim = 64, num_layers = 4 ─────────────────────────────
model_32_h64_nl4, diff_32_h64_nl4 = run_experiment(
    window_length=32, num_epochs=2000, hidden_dim=64, num_layers=4, eval_every=500)


StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H64-NL4-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H64-NL4-E2000] params: 623,878
[decompdiff-L32-H64-NL4-E2000] ep    1/2000  train=1.19545  lr=5.80e-05
[decompdiff-L32-H64-NL4-E2000] ep    2/2000  train=0.62655  lr=1.00e-04
[decompdiff-L32-H64-NL4-E2000] ep    3/2000  train=0.24410  lr=1.00e-04
[decompdiff-L32-H64-NL4-E2000] ep    4/2000  train=0.20802  lr=1.00e-04
[decompdiff-L32-H64-NL4-E2000] ep    5/2000  train=0.18432  lr=1.00e-04
[decompdiff-L32-H64-NL4-E2000] ep    6/2000  train=0.16834  lr=1.00e-04
[decompdiff-L32-H64-NL4-E2000] ep    7/2000  train=0.15815  lr=1.00e-04
[decompdiff-L32-H64-NL4-E2000] ep    8/2000  train=0.14904  lr=1.00e-04
[decompdiff-L32-H64-NL4-E2000] ep    9/2000  train=0.14568  lr=1.00e-04
[decompdiff-L32-H64-NL4-E2000] ep   10/2000  train=0.14165  lr=1.00e-04
[decompdiff-L32-H64-NL4-E2000] ep   11/2000  train=0.13893  lr=1.00e-04
[decompdiff-L32-H64-NL4-

correlational_score,▁▄█▄
disc_score,▁▇█▇
disc_score_std,▁▄▄█
epoch,▁▁▁▁▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇████
fdds,▁▇▆█
lr,██████████▇▇▆▆▆▅▅▅▅▅▄▄▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
pred_mae,▁▆█▇
pred_mae_std,▁█▇▁
test_acc,▁▇█▇
train_loss,█▆▅▆▆▅▄▅▃▄▃▃▄▄▃▃▃▃▂▂▃▃▃▂▄▃▃▂▃▂▁▂▂▂▂▂▃▂▃▁
+1,...
